# Submission 4: Project Development Practices (5%)

**Course:** RBB2013 / FFM2063 / FEM2063, Digital Twin, May 2026
**Project:** SmartClean Twin, a software-emulated Digital Twin of a mobile
inspection and cleaning robot (project topic 2)
**Repository:** https://github.com/KAI-UTP/smartclean-twin
**Presentation & demo video:** [https://youtu.be/zEq7L-ivMLA](https://youtu.be/zEq7L-ivMLA)

**Team Members**

| No | Name | Student ID |
|---|---|---|
| 1 | Chan Li Kai | 22010900 |
| 2 | William Wong Xiao Kang | 22010943 |
| 3 | Irvin Chang Hou Ceng | 22012342 |
| 4 | Liang Yan Ee | 22011522 |
| 5 | Nurin Emelin Binti Marhisyam | 24006706 |

> **How to reproduce:** start the stack with `docker compose up -d` (8 containers),
> then run this notebook top to bottom. All outputs below were produced against
> the running system.


## 1. Executive Summary

The SmartClean Twin was developed using agile practice with two sprint cycles,
Git version control on a shared `main` branch, a four-level automated test suite,
and a CI pipeline that lints, tests and builds on every push.

| Practice | Evidence in this submission |
|---|---|
| Agile sprint planning & execution | Product backlog of 10 user stories, 31 tasks across 2 sprints, with sprint goals, acceptance criteria and reviews (Sections 2-4) |
| Deliverables from every team member | Per-member task and artefact table, each committed from that member's own GitHub account (Section 5) |
| Version control & group merge | Shared `main`, rebase-based integration, CI gate; distinct commit authors listed live (Sections 6, 11) |
| Unit tests for each module | 81 unit tests mapped module by module (Section 8) |
| Integration / interface tests | Command API ↔ MQTT ↔ simulator, ingestion ↔ InfluxDB (Section 8) |
| System tests | Full twin flow, fault scenarios, persistence (Section 8) |
| Regression suite | 10 golden-snapshot tests run automatically in CI (Sections 8, 9) |
| Demonstrated pass **and fail** cases | Live experiment in Section 10: broker stopped → tests fail → broker restored → tests pass |
| Automatic build & deployment | GitHub Actions: lint → tests → AI model training → 5 Docker image builds → compose validation (Section 9) |

Totals verified live in Section 9: **81 unit, 11 integration, 14 system and
10 regression tests, all passing.**


## 2. Agile Approach

### 2.1 Why agile for a Digital Twin

A Digital Twin is not fully specifiable in advance: what the twin should display
and predict only becomes clear once real telemetry is flowing. The project was
therefore organised into two short sprints, each ending in a **working,
deployable system** rather than a document. Sprint 1 delivered the complete data
path; Sprint 2 added intelligence and visualization on top of it. This ordering
was intentional, the AI service in Sprint 2 was able to subscribe to a topic
that already existed and was already carrying validated data, so it required no
change to any Sprint 1 component.

### 2.2 Product backlog

Ten user stories were written from the operator's and engineer's points of view
and estimated in story points:

| ID | User story | Priority | Points |
|---|---|---|---|
| US-01 | Operator: see live robot telemetry in Grafana | High | 8 |
| US-02 | Operator: obstacle emergencies detected automatically | High | 5 |
| US-03 | Operator: send PAUSE/RESUME/STOP and receive acknowledgement | High | 8 |
| US-04 | Operator: watch cleaning coverage increase toward target | High | 5 |
| US-05 | Operator: battery and motor alerts before failure | Medium | 5 |
| US-06 | Engineer: data persists after container restart | High | 3 |
| US-07 | Engineer: AI-predicted motor health shown in Grafana | Medium | 8 |
| US-08 | Engineer: unit and integration tests for every module | High | 8 |
| US-09 | Engineer: CI pipeline runs tests on every push | Medium | 3 |
| US-10 | Engineer: scale the ingestion service without data loss | Low | 3 |

Full backlog, task breakdown, acceptance criteria, definition of done and sprint
reviews are in `docs/sprint-plan.md`.


## 3. Sprint 1: Core Data Flow (1-11 July 2026)

**Sprint goal.** End-to-end telemetry flows from the robot simulator to the
validated MQTT topic, Digital Twin state is computed, and commands return
acknowledgements, all runnable with a single `docker compose up`.

**Scope delivered (T-01 … T-15):** shared Pydantic models and MQTT topics; JSON
schema contracts; grid map and lawnmower path; robot physics and telemetry
publisher; command handler with ACK; Mosquitto configuration; telemetry
ingestion with validation and InfluxDB write; the state engine and its rules;
the Command API; unit tests for schema, rules, grid map and commands;
integration tests for ingestion and the Command API; Dockerfiles for all five
custom services; and `docker-compose.yml`. Reviews by William (T-14) and Irvin
(T-15).

**Acceptance criteria, all met:** telemetry published every 1 s; invalid
messages rejected by ingestion; all 11 state variables produced;
`OBSTACLE_EMERGENCY` raised when `obstacle_cm < 25`; PAUSE acknowledged within
2 s; `docker compose up` starts every service without error; unit tests pass
locally.

**Sprint review outcome.** The data path was sound, but two things were learned
that shaped Sprint 2: the dashboard needed derived state, not only raw sensor
values, to be useful to an operator; and the simulator originally stopped after
one cleaning pass, which made long-running observation impossible.

## 4. Sprint 2: Intelligence, Visualization and Robustness (12-19 July 2026)

**Sprint goal.** Add the AI layer, the visualization layer, persistence proof,
scaling and CI, and harden the system for continuous operation.

**Scope delivered (T-16 … T-31):** the AI training pipeline and AI
microservice; InfluxDB auto-initialisation; Grafana datasource and dashboard
provisioning; the persistence test; system tests for the full flow and the
fault scenarios; the regression suite; the GitHub Actions CI workflow; smoke and
demo-data scripts; all documentation; and the NVIDIA Omniverse 3D scene
(T-31). Reviews by Liang (T-27, T-28) and Nurin (T-29); final demonstration
preparation by the whole team (T-30).

**Acceptance criteria, all met:** AI accuracy ≥ 80 % on held-out data (achieved
90.8 % for health state, R² 0.91 for RUL); AI predictions visible in Grafana;
persistence test passes after an InfluxDB restart; `docker compose up --scale
telemetry-ingestion=2` runs without port conflict; all 10 regression tests pass;
CI runs lint, tests and Docker build on push.

**Issues found and fixed during the sprint** (recorded because they are the real
evidence that the practices worked):

| Issue | How it was caught | Resolution |
|---|---|---|
| Simulator stopped after one cleaning pass | Long-running observation | Path index resets; floor re-dirties, enabling continuous operation |
| 3D tile index wrong for non-integer coordinates | Manual verification against a known cell | Index computed as `int(x / CELL_SIZE_M)` |
| Twin state not reaching the 3D scene | Robot stayed green during an EMERGENCY | Flux query grouped and pivoted so all fields arrive together |
| Persistence test reported false data loss | Test failed while the system was healthy | Frozen query window instead of a sliding one |
| Scaling to 2 ingestion replicas failed | Scaling demonstration | Host port range instead of a fixed host port |
| CI red on documentation-only commits | Teammate's push | Lint tool versions pinned to match the development environment |


## 5. Deliverables from Every Team Member

Tasks are assigned per member in `docs/sprint-plan.md`. Every member delivered
their artefact **committed from their own GitHub account**, so authorship is
independently verifiable in the version-control history (Section 11).

| Member | Sprint tasks | Deliverable committed | GitHub account |
|---|---|---|---|
| Chan Li Kai | T-01 to T-13, T-16 to T-26, T-31 | All service code, test suites, Dockerfiles, CI workflow, Grafana dashboard, Omniverse scene, documentation | KAI-UTP |
| William Wong Xiao Kang | T-14, review MQTT topics and the telemetry contract | `docs/review-william.md`, end-to-end message trace verified hop by hop against the contract | williamwxk0822 |
| Irvin Chang Hou Ceng | T-15, review state rules and AI test cases | `docs/review-irvin.md` plus three what-if scenario evidence screenshots in `docs/evidence/` | vinutp |
| Liang Yan Ee | T-27, T-28, verify test suites, dashboard panels and command/ACK flow | `docs/review-liang.md`, test verification report | renerere |
| Nurin Emelin Binti Marhisyam | T-29, dashboard review and sprint evidence collection | `docs/review-nurin.md` plus dashboard and fault-injection evidence screenshots | nrnemelin |

## 6. Version Control and Group Merge

**Repository.** A single GitHub repository with one shared long-lived branch,
`main`.

**Workflow.** Commits are small and descriptive, and reference the task or the
defect they address. Because five people push to one branch, integration is done
by **rebase** (`git pull --rebase origin main`) rather than merge commits, which
keeps the history linear and readable, an important property when the history
itself is submitted as evidence.

**Group merge at end of sprint.** Each member pushed their own deliverable into
`main` at the end of the sprint in which it was assigned. Conflicts were resolved
by rebasing onto the latest `main` before pushing.

**Branch protection by CI.** Every push triggers the pipeline in Section 9. A
failing lint or test run marks the commit red, so a broken state on `main` is
immediately visible to the whole team. This mechanism did its job during the
project: a documentation-only push from a teammate went red because the lint
tools had auto-updated to a stricter version, which led to pinning the tool
versions.


## 7. Test-Driven Approach

Tests were written alongside each module rather than after the system was
complete, and the suite is structured as a pyramid so that the fastest, most
localised tests run first:

| Level | Count | Runtime | Requires running stack? | What it establishes |
|---|---|---|---|---|
| Unit | 81 | ~1 s | No | Each function behaves correctly in isolation |
| Integration | 11 | ~2 s | Partly | Two components interact correctly across their interface |
| System | 14 | ~25 s | Yes | The whole twin behaves correctly end to end |
| Regression | 10 | ~1 s | No | Previously correct behaviour has not drifted |

The pyramid shape is deliberate: unit tests are numerous and cheap so they can
run on every save and in every CI job, while system tests are few and expensive
and require the full compose stack. System tests are guarded by an
`INTEGRATION_TEST=1` environment variable so that a developer without Docker
running still gets a meaningful, fast test run instead of a wall of errors.


## 8. Tests for Each Module

Every module has unit tests, and every interface between modules has an
integration or system test:

| Module | Unit tests | Integration / interface tests |
|---|---|---|
| `shared/smartclean_common`, schemas, topics | `tests/unit/test_telemetry_schema.py` (17 tests: field presence, type coercion, range limits, ISO-8601 timestamps, robot-id constraints) | Exercised by every integration and system test, since all services import these models |
| `robot-simulator`, physics, grid, commands | `tests/unit/test_grid_map.py` (10), `tests/unit/test_simulator_commands.py` (12) | `tests/system/test_full_flow.py`, command accepted and acknowledged |
| `state-engine`, rules, twin state | `tests/unit/test_state_rules.py` (20: threshold boundaries for each state variable and alarm) | `tests/system/test_full_flow.py`, telemetry in, correct state out |
| `ai-service`, models, predictor | `tests/unit/test_ai_predictor.py`, prediction shape, fallback path | `tests/system/test_full_flow.py`, predictions published and stored |
| `command-api`, validation, MQTT publish | `tests/unit/test_command_validation.py`, invalid robot id and invalid command rejected | `tests/integration/test_command_api.py`, REST → MQTT → ACK round trip |
| `telemetry-ingestion`, validate, store, republish | Covered through the shared-schema unit tests | `tests/integration/test_telemetry_ingestion.py`, valid message stored, invalid message rejected and counted |
| Whole system |, | `tests/system/` (14) and `tests/regression/` (10) |

### 8.1 Fault-scenario system tests

Three of the system tests inject a fault and assert that the twin reaches the
correct state: obstacle → `EMERGENCY`, motor overload → degraded motor health,
low battery → battery state and alarm. These are the tests that verify the twin
*interprets* the asset correctly, not merely that data moves.


## 9. CI/CD, Automatic Build and Regression on Every Push

`.github/workflows/ci.yml` defines five jobs with an explicit dependency graph,
so an early cheap failure prevents expensive work from running:

```
lint-and-format ──┬─► unit-tests ──► build-docker-images ──► validate-compose
                  └─► train-ai-model
```

| Job | Action | Purpose |
|---|---|---|
| Lint & format | `ruff check`, `black --check` (versions pinned) | Style and static errors caught before tests |
| Unit tests | `pytest` over unit, integration and regression suites with coverage | Functional regression gate |
| Train & validate AI model | Runs `train_model.py` | Model quality is a build gate: the script exits non-zero if accuracy or R² falls below target |
| Build Docker images | Builds all six service images | Guarantees every service is still deployable |
| Validate compose | `docker compose config --quiet` | Catches invalid orchestration configuration |

Because AI training runs in CI, a change that degrades model performance fails
the build in the same way a broken function would. Model artefacts therefore
cannot drift away from the code that produced them.

### 9.1 Live evidence, full test suite, all four levels

In [1]:
import os, subprocess, sys

env = dict(os.environ, INTEGRATION_TEST="1")
totals = {}
for label, target in [
    ("Unit", "tests/unit"),
    ("Integration", "tests/integration"),
    ("System", "tests/system"),
    ("Regression", "tests/regression"),
]:
    r = subprocess.run([sys.executable, "-m", "pytest", target, "-q", "--no-header"],
                       capture_output=True, text=True, cwd=".", env=env)
    summary = [l for l in r.stdout.splitlines() if l.strip()][-1:]
    totals[label] = r.returncode
    print(f"{label:12s} {summary[0] if summary else 'no output':62s} exit={r.returncode}")

print()
print("All four levels passing:", all(v == 0 for v in totals.values()))


Unit         ============================= 81 passed in 0.82s ============================== exit=0


Integration  ============================= 11 passed in 2.27s ============================== exit=0


System       ============================= 14 passed in 20.65s ============================= exit=0


Regression   ============================= 10 passed in 0.90s ============================== exit=0

All four levels passing: True


## 10. Demonstrated FAIL Case and Recovery

A test suite that has never failed is not evidence of anything: it may simply be
insensitive. The rubric requires demonstrated pass **and fail** cases, so the
experiment below deliberately breaks the system and shows the suite detecting it.

**Procedure.**

1. Run the system tests with the stack healthy, they must **pass**.
2. Stop the MQTT broker container, removing the message bus that the twin
   depends on, the tests must now **fail**.
3. Restart the broker and wait for the services to reconnect, the same tests
   must **pass** again, with no code change.

Step 3 additionally demonstrates fault tolerance: the services reconnect to the
broker automatically with exponential back-off, so the system self-heals once
the dependency returns.

In [2]:
import os, subprocess, sys, time

env = dict(os.environ, INTEGRATION_TEST="1")

def run_system_tests(label):
    r = subprocess.run([sys.executable, "-m", "pytest",
                        "tests/system/test_full_flow.py", "-q", "--no-header"],
                       capture_output=True, text=True, cwd=".", env=env)
    tail = [l for l in r.stdout.splitlines() if l.strip()][-1:]
    print(f"{label}: {tail[0] if tail else r.stdout[-200:]}   (exit {r.returncode})")
    return r.returncode

print("STEP 1: stack healthy, tests expected to PASS")
rc_before = run_system_tests("  result ")

print("\nSTEP 2: deliberately stop the MQTT broker; tests expected to FAIL")
subprocess.run(["docker", "stop", "smartclean-mosquitto"], capture_output=True, cwd=".")
time.sleep(5)
rc_broken = run_system_tests("  result ")

print("\nSTEP 3: restart the broker; tests expected to PASS again")
subprocess.run(["docker", "start", "smartclean-mosquitto"], capture_output=True, cwd=".")
time.sleep(20)
rc_fixed = run_system_tests("  result ")

print()
print(f"Baseline passed          : {rc_before == 0}")
print(f"Fail case demonstrated   : {rc_broken != 0}")
print(f"Recovery demonstrated    : {rc_fixed == 0}")


STEP 1: stack healthy, tests expected to PASS


  result : ============================= 11 passed in 9.14s ==============================   (exit 0)

STEP 2: deliberately stop the MQTT broker; tests expected to FAIL


  result : ======================== 3 failed, 8 passed in 29.19s =========================   (exit 1)

STEP 3: restart the broker; tests expected to PASS again


  result : ============================= 11 passed in 9.09s ==============================   (exit 0)

Baseline passed          : True
Fail case demonstrated   : True
Recovery demonstrated    : True


## 11. Live Evidence, Version Control History

The history below shows commits from every team member's own account, which is
the evidence for individual contribution and group merge into the shared
branch.

In [3]:
import subprocess

r = subprocess.run(["git", "log", "--format=%h|%an|%ad|%s", "--date=short", "-18"],
                   capture_output=True, text=True, cwd=".")
for line in r.stdout.splitlines():
    parts = line.split("|")
    if len(parts) == 4:
        h, an, ad, s = parts
        print(f"{h}  {an[:26]:28s} {ad}  {s[:50]}")

print()
print("Distinct commit authors (group merge evidence):")
a = subprocess.run(["git", "log", "--format=%an"], capture_output=True, text=True, cwd=".")
for name in sorted(set(a.stdout.split(chr(10))) - {""}):
    print("  -", name)

print()
n = subprocess.run(["git", "rev-list", "--count", "HEAD"],
                   capture_output=True, text=True, cwd=".")
print("Total commits on main:", n.stdout.strip())


4a2c073  KAI-UTP                      2026-08-11  docs: numbered code walkthrough for the live prese
9f71be7  KAI-UTP                      2026-07-27  docs: correct Sprint 2 window to match the actual 
9593702  KAI-UTP                      2026-07-27  docs: replace discussion questions with a limitati
e92761e  KAI-UTP                      2026-07-27  tools: WebPDF and printable-HTML export scripts
605cf54  KAI-UTP                      2026-07-27  tools: add notebooks_to_pdf.py, PDF export without
2a4dc2e  renerere                     2026-07-27  Add files via upload
f26c326  KAI-UTP                      2026-07-27  style: replace em and en dashes with plain punctua
9383ab3  KAI-UTP                      2026-07-27  docs: rewrite all five submission notebooks with f
f273895  KAI-UTP                      2026-07-27  docs: fix walkthrough notebook title
76664b2  KAI-UTP                      2026-07-27  docs: close remaining rubric gaps in dev-practices
18e8b77  KAI-UTP                     

## 12. Discussion and Limitations

**What the practices achieved.** The two-sprint structure meant a working system
existed from the end of Sprint 1, so Sprint 2 could be spent on value rather
than integration. The test suite repeatedly caught real defects, the wrong 3D
tile index, twin state not reaching the scene, a false-positive persistence
failure, each of which would have been invisible in a manual demonstration.
The CI gate caught a lint-version drift that would otherwise have quietly
blocked the whole team.

**Limitations, stated honestly.**

1. **Contribution is not evenly distributed.** One member wrote the code; the
   other four contributed review, verification, evidence and documentation.
   This is recorded transparently in Section 5 rather than presented as equal
   authorship.
2. **Tests were written alongside modules, not strictly test-first.** The
   discipline is test-driven in spirit, no module was considered done without
   tests, but tests did not always precede implementation.
3. **No pull-request review workflow.** Work was pushed to `main` behind a CI
   gate rather than through pull requests with mandatory review. For a larger
   team, PR review would be the next step.
4. **Coverage is not uniform.** The state engine and shared schemas are heavily
   tested; the AI predictor has the thinnest unit coverage because most of its
   behaviour is validated statistically during training instead.
5. **CD stops at build.** The pipeline lints, tests, trains and builds images,
   but does not push them to a registry or deploy to an environment, since the
   deployment target is a local Docker host.


## 13. Conclusion

Development followed agile practice across two sprints with a ten-story backlog,
31 tracked tasks, per-member deliverables and documented sprint reviews. Version
control is a single shared branch integrated by rebase, protected by a CI gate,
with commits from every team member's own account. The automated suite spans four
levels, 81 unit, 11 integration, 14 system and 10 regression tests, all
verified passing live in Section 9, and Section 10 demonstrates the suite
correctly failing when the message broker is removed and passing again once it
returns. CI runs lint, the full suite, AI model training with a performance gate,
and all six Docker image builds on every push.
